In [11]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [12]:
import os
import random
import torch
import torchaudio
import numpy as np


from torch.utils.data import Dataset
from transformers import (
Wav2Vec2Processor,
Wav2Vec2ForSequenceClassification,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)


from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import classification_report

In [13]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42


In [14]:
# -------------------------
# Mapeo de emociones (3 clases)
# -------------------------
emotion2id = {
    "happiness": 0,      # Positivo
    "neutral": 1,      # Neutral
    "sadness": 2,     # Negativo
    "anger": 2,
    "disgust": 2,
    "fear": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}


In [15]:
# -------------------------
# Dataset
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        emotion = filename.split("_")[0].lower()
        label = emotion2id[emotion]
        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        speech = speech.mean(dim=0)  # mono

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }


In [16]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    padding=True,
    return_tensors="pt"
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(DEVICE)

# Congelar extractor (clave para corpus pequeño)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
projector.weight  | MISSING    | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 
projector.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# -------------------------
# División del corpus
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]
random.shuffle(all_files)

train_size = int(0.8 * len(all_files))
train_files = all_files[:train_size]
test_files = all_files[train_size:]

print(f"Train: {len(train_files)} | Test: {len(test_files)}")

train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)


Train: 689 | Test: 173


In [18]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }


In [19]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
# -------------------------
# Entrenamiento y evaluación
# -------------------------
trainer.train()

results = trainer.evaluate()
print("\nResultados finales:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# -------------------------
# Baseline Aleatoria
# -------------------------
from sklearn.metrics import classification_report

# Obtener etiquetas reales del test set
true_labels = []

for i in range(len(test_dataset)):
    true_labels.append(test_dataset[i]["labels"].item())

true_labels = np.array(true_labels)

# Generar predicciones aleatorias
random_preds = np.random.randint(0, NUM_LABELS, size=len(true_labels))

# Calcular métricas
rand_precision, rand_recall, rand_f1, _ = precision_recall_fscore_support(
    true_labels,
    random_preds,
    average="weighted",
    zero_division=0
)

rand_acc = accuracy_score(true_labels, random_preds)

print("\nResultados Baseline Aleatoria:")
print(f"Accuracy: {rand_acc:.4f}")
print(f"Precision: {rand_precision:.4f}")
print(f"Recall: {rand_recall:.4f}")
print(f"F1-score: {rand_f1:.4f}")

# Fijar semillas


random.seed(SEED)
torch.manual_seed(SEED)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,2.487305,1.145493,0.658960,0.573558,0.658960,0.555023
2,0.054372,0.852372,0.751445,0.747256,0.751445,0.740118
3,1.010580,2.128134,0.572254,0.702459,0.572254,0.571882
4,1.351291,0.940452,0.803468,0.818605,0.803468,0.799765
5,0.005665,1.521571,0.710983,0.818160,0.710983,0.726706
6,0.005074,0.887599,0.849711,0.865046,0.849711,0.846563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,2.487305,1.145493,0.658960,0.573558,0.658960,0.555023
2,0.054372,0.852372,0.751445,0.747256,0.751445,0.740118
3,1.010580,2.128134,0.572254,0.702459,0.572254,0.571882
4,1.351291,0.940452,0.803468,0.818605,0.803468,0.799765
5,0.005665,1.521571,0.710983,0.818160,0.710983,0.726706
6,0.005074,0.887599,0.849711,0.865046,0.849711,0.846563
7,1.290901,0.897591,0.843931,0.868048,0.843931,0.839336
8,0.316879,0.814464,0.867052,0.877690,0.867052,0.863941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Resultados finales:
eval_loss: 0.8145
eval_accuracy: 0.8671
eval_precision: 0.8777
eval_recall: 0.8671
eval_f1-score: 0.8639
eval_runtime: 4.5033
eval_samples_per_second: 38.4160
eval_steps_per_second: 38.4160
epoch: 8.0000

Resultados Baseline Aleatoria:
Accuracy: 0.3526
Precision: 0.4500
Recall: 0.3526
F1-score: 0.3724
